# 🎬 AnimeEncoderBot
**GPU-accelerated video encoding (AV1/HEVC) + AI anime upscaling**

⚠️ Make sure GPU T4 is enabled: **Settings → Accelerator → GPU T4 x2**

Just hit **Run All** — everything is automated.

In [ ]:
# ═══ Step 1: Check GPU ═══
!nvidia-smi
print('\n' + '='*60)
!ffmpeg -version 2>/dev/null | head -1 || echo 'FFmpeg not found'
print('='*60)

In [ ]:
# ═══ Step 2: Load Secrets ═══
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

os.environ['BOT_TOKEN']       = secrets.get_secret('BOT_TOKEN')
os.environ['API_ID']          = secrets.get_secret('API_ID')
os.environ['API_HASH']        = secrets.get_secret('API_HASH')
os.environ['ADMIN_IDS']       = secrets.get_secret('ADMINS')
os.environ['LOG_CHANNEL']     = secrets.get_secret('LOG_CHANNEL')
os.environ['GDRIVE_FOLDER_ID']= secrets.get_secret('GDRIVE_FOLDER_ID')

# GDrive SA JSON — write to file
sa_json = secrets.get_secret('GDRIVE_SA_JSON')
with open('/kaggle/working/sa.json', 'w') as f:
    f.write(sa_json)
os.environ['GDRIVE_SA_JSON'] = '/kaggle/working/sa.json'

# MongoDB — use local MongoDB for Kaggle
os.environ['MONGO_URI']       = 'mongodb://localhost:27017/anime_encoder_bot'
os.environ['GPU_ENABLED']     = 'true'
os.environ['CONCURRENT_TASKS']= '2'
os.environ['DOWNLOAD_DIR']    = '/kaggle/working/downloads'

!mkdir -p /kaggle/working/downloads
print('✅ Secrets loaded')

In [ ]:
# ═══ Step 3: Install MongoDB & System Dependencies ═══
import subprocess
import os
import glob

cmds = [
    'apt-get update -qq',
    'apt-get install -y -qq gnupg curl libvulkan1 vulkan-tools libgomp1',
]
for cmd in cmds:
    subprocess.run(cmd, shell=True, capture_output=True)
print('📦 Base packages and Vulkan dependencies installed')

# Configure NVIDIA Vulkan ICD — auto-detect the correct library path
# On Kaggle containers, libGLX_nvidia.so.0 often doesn't work.
# We search for the actual libnvidia-vulkan-producer.so or libGLX_nvidia.so
!mkdir -p /etc/vulkan/icd.d

nvidia_vulkan_lib = None
search_paths = [
    '/usr/lib/x86_64-linux-gnu/libnvidia-vulkan-producer.so',
    '/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.0',
    '/usr/lib/libnvidia-vulkan-producer.so',
    '/usr/lib/libGLX_nvidia.so.0',
]
# Also search by glob for versioned paths
for pattern in ['/usr/lib/x86_64-linux-gnu/libnvidia-vulkan-producer.so*',
                '/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so*']:
    search_paths.extend(glob.glob(pattern))

for lib_path in search_paths:
    if os.path.exists(lib_path):
        nvidia_vulkan_lib = lib_path
        break

if nvidia_vulkan_lib:
    import json
    icd = {"file_format_version": "1.0.0", "ICD": {"library_path": nvidia_vulkan_lib, "api_version": "1.3"}}
    with open('/etc/vulkan/icd.d/nvidia_icd.json', 'w') as f:
        json.dump(icd, f)
    print(f'✅ NVIDIA Vulkan ICD registered: {nvidia_vulkan_lib}')
else:
    # Fallback: try installing the NVIDIA Vulkan ICD package
    drv = subprocess.run('cat /proc/driver/nvidia/version | head -1',
                        shell=True, capture_output=True, text=True).stdout
    print(f'⚠️ No NVIDIA Vulkan library found. Driver info: {drv.strip()}')
    print('  Attempting to install libnvidia-gl...')
    subprocess.run('apt-get install -y -qq libnvidia-gl-* 2>/dev/null || true', shell=True, capture_output=True)
    # Write a best-effort ICD
    !echo '{"file_format_version":"1.0.0","ICD":{"library_path":"libGLX_nvidia.so.0","api_version":"1.3"}}' > /etc/vulkan/icd.d/nvidia_icd.json
    print('⚠️ NVIDIA Vulkan ICD registered (best-effort — Real-ESRGAN may fall back to CPU)')

# Verify Vulkan sees the GPU
vk_check = subprocess.run('vulkaninfo --summary 2>/dev/null | grep -i gpu || echo "No Vulkan GPU detected"',
                          shell=True, capture_output=True, text=True)
print(f'🔍 Vulkan check: {vk_check.stdout.strip()}')

# Add MongoDB repo
!curl -fsSL https://www.mongodb.org/static/pgp/server-7.0.asc | gpg --dearmor -o /usr/share/keyrings/mongodb-server-7.0.gpg 2>/dev/null
!echo 'deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse' > /etc/apt/sources.list.d/mongodb-org-7.0.list
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y -qq mongodb-org > /dev/null 2>&1
!mkdir -p /data/db
!mongod --fork --logpath /var/log/mongod.log --dbpath /data/db
print('✅ MongoDB running')

In [ ]:
# ═══ Step 4: Install Real-ESRGAN ═══
import os

REALESRGAN_DIR = '/kaggle/working/realesrgan'
os.environ['REALESRGAN_PATH'] = f'{REALESRGAN_DIR}/realesrgan-ncnn-vulkan'

if not os.path.exists(f'{REALESRGAN_DIR}/realesrgan-ncnn-vulkan'):
    !mkdir -p {REALESRGAN_DIR}
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesrgan-ncnn-vulkan-20220424-ubuntu.zip -O /tmp/realesrgan.zip
    !unzip -o /tmp/realesrgan.zip -d {REALESRGAN_DIR} > /dev/null
    !chmod +x {REALESRGAN_DIR}/realesrgan-ncnn-vulkan
    !rm /tmp/realesrgan.zip
    print('✅ Real-ESRGAN installed')
else:
    print('✅ Real-ESRGAN already installed')

In [ ]:
# ═══ Step 5: Clone Bot & Install Dependencies ═══
import os

# Clone from GitHub
!git clone https://github.com/Alaxroy121/AnimeEncoderBot.git /kaggle/working/bot

# Install Python dependencies
!pip install -q pyrogram tgcrypto motor pymongo python-dotenv aiofiles aiohttp google-api-python-client google-auth
print('✅ Dependencies installed')

In [ ]:
# ═══ Step 6: Start the Bot ═══
import os
os.chdir('/kaggle/working/bot')

# Verify all files are present
required = ['bot.py', 'commands.py', 'callbacks.py', 'encoder.py',
            'upscaler.py', 'database.py', 'queue_manager.py',
            'utils.py', 'config.py', 'gdrive.py']
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print(f'❌ Missing files: {missing}')
    print('Upload them before running!')
else:
    print('✅ All files present. Starting bot...')
    !python3 bot.py

In [ ]:
# ═══ Keep-Alive (run separately if needed) ═══
import time
from IPython.display import clear_output

i = 0
while True:
    i += 1
    time.sleep(300)
    clear_output(wait=True)
    print(f'🔄 Keep-alive #{i} — session active')